In [ ]:
# Завдання 1. Визначення персони та тональності висловлювання

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI


# 1. Завантаження ключа з .env у корені проєкту.
# Ноутбук hw04.ipynb відкриваємо з папки проєкту.
env_path = Path.cwd() / ".env"

if not env_path.is_file():
    raise FileNotFoundError(f"Не знайдено файл .env у папці: {Path.cwd()}")

load_dotenv(dotenv_path=env_path)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("Не знайдено OPENROUTER_API_KEY у змінних середовища або .env.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    timeout=60.0,
    max_retries=0,
)

MODEL = "openai/gpt-oss-120b"


# 2. Дані для аналізу.
texts = [
    "Оксана Дрозд натхненно провела демо — команда аплодувала стоячи.",
    "Керівник відділу Андрій Король грубо порушив інструкції, через що реліз зірвано.",
    "Роман Бондаренко надіслав звіт о 14:00 і підтвердив графік зустрічей.",
    "Сьогодні Ілля Ткачук блискуче оптимізував код і заощадив 30% бюджету.",
    "Марко Савченко ігнорує повідомлення й затримує погодження — це неприйнятно.",
]


# 3. Інструкція для моделі.
PROMPT = """
ПРацюй,як система вилучення імен людей і класифікації тональності
українськомовних повідомлень.

Завдання:
У вхідному тексті згадано рівно одну людину.
Визнач її ім'я та прізвище і тональність висловлювання саме щодо неї.

Правила для person:
- Відтвори ім'я та прізвище точно так, як вони записані у вхідному тексті.
- Збережи написання, регістр і порядок слів.
- Не додавай посаду, титул, звертання або інші уточнення.
- Не вигадуй відсутніх частин імені.

Правила для tone:
- "позитивна": похвала, схвалення, захоплення або позитивна оцінка
  дій чи результатів людини.
- "негативна": критика, осуд, невдоволення або негативна оцінка
  дій чи результатів людини.
- "нейтральна": фактичне повідомлення без вираженого схвалення чи осуду.
Оцінюй ставлення до людини, а не лише загальний емоційний фон тексту.
Не додавай оцінку від себе.

Правила формату:
- Поверни рівно один JSON-об'єкт із рівно двома ключами: "person" і "tone".
- Обидва значення мають бути рядками.
- Значення tone — лише "позитивна", "нейтральна" або "негативна".
- Використовуй подвійні лапки.
- Не додавай Markdown, блоки коду, коментарі, пояснення або інші ключі.
- Не виводь міркувань. Виведи лише кінцевий JSON-об'єкт.


Приклади:

Текст: <<<Олена Коваль чудово підготувала презентацію.>>>
Відповідь: {"person":"Олена Коваль","tone":"позитивна"}

Текст: <<<Менеджер Петро Мельник недбало виконав роботу й отримав догану.>>>
Відповідь: {"person":"Петро Мельник","tone":"негативна"}

Текст: <<<Наталія Мороз надіслала документ о 10:00.>>>
Відповідь: {"person":"Наталія Мороз","tone":"нейтральна"}
""".strip()


# 4. Виклик моделі для кожного тексту та вивід результатів.
for t in texts:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": PROMPT.strip()},
            {"role": "user",
             "content": f"Проаналізуй текст і поверни ТІЛЬКИ JSON з ключами person та tone.\\nТекст: <<<{t}>>>"},
        ],
        temperature=0
    )
    result = resp.choices[0].message.content
    json.loads(result) # перевірка, чи валідний JSON
    print(result)

{"person":"Оксана Дрозд","tone":"позитивна"}
{"person":"Андрій Король","tone":"негативна"}
{"person":"Роман Бондаренко","tone":"нейтральна"}
{"person":"Ілля Ткачук","tone":"позитивна"}
{"person":"Марко Савченко","tone":"негативна"}


In [16]:
# Завдання 2. Тріаж відгуків: визначення продукту та пріоритету

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI


# 1. Завантаження ключа 

env_path = Path.cwd() / ".env"

if not env_path.is_file():
    raise FileNotFoundError(f"Не знайдено файл .env у папці: {Path.cwd()}")

load_dotenv(dotenv_path=env_path)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError(
        "Не знайдено OPENROUTER_API_KEY у змінних середовища або .env."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    timeout=60.0,
    max_retries=0,
)

MODEL = "openai/gpt-oss-120b"


# 2. Дані для аналізу
data = [
    {
        "text": "Квитанції не завантажуються в Atlas Pro з мобільного — фінвідділ стоїть.",
        "product": "Atlas",
        "priority": "високий",
    },
    {
        "text": "Після оновлення 3.1 у Core графіки будуються на 2 секунди довше, але працюють стабільно.",
        "product": "Core",
        "priority": "середній",
    },
    {
        "text": "Автовідповідь у Desk на свята — nice-to-have, допоможе трохи зменшити навантаження на операторів.",
        "product": "Desk",
        "priority": "низький",
    },
    {
        "text": "Core падає під час імпорту CSV >100 тис. рядків — роботу заблоковано.",
        "product": "Core",
        "priority": "високий",
    },
    {
        "text": "Atlas показує старі курси валют, поки що оновлюємо вручну.",
        "product": "Atlas",
        "priority": "високий",
    },
]


# 3. Інструкція для моделі.
PROMPT = """
Працюй як система тріажу українськомовних відгуків
про програмні продукти.

Завдання:
Визнач продукт, згаданий у відгуку, та пріоритет звернення.
Кожен відгук стосується одного продукту.

Правила для product:
- Допустимі значення: лише "Atlas", "Core", "Desk".
- Зберігай точне написання та регістр цих назв.
- Назви редакцій нормалізуй до базового продукту:
  "Atlas Pro" потрібно повертати як "Atlas".
- Не додавай назву редакції, номер версії або інші уточнення.

Правила для priority:
- "високий": роботу заблоковано, важлива операція не виконується,
  продукт падає або відображає неправильні чи застарілі фінансові дані.
  Застарілі курси валют мають високий пріоритет навіть тоді,
  коли користувачі тимчасово оновлюють їх вручну.
- "середній": наявна функціональність або швидкодія погіршилася,
  але роботу можна продовжувати, результати залишаються коректними
  і немає ознак високого пріоритету.
- "низький": необов'язкове покращення, побажання або nice-to-have
  без повідомлення про несправність чи погіршення функціональності.

Якщо є кілька сигналів, обирай пріоритет:
"високий" важливіший за "середній",
"середній" важливіший за "низький".
Враховуй контекст і заперечення.
Не вигадуй проблем або наслідків, яких немає у відгуку.

Правила формату:
- Поверни рівно один JSON-об'єкт із рівно двома ключами:
  "product" і "priority".
- Обидва значення мають бути рядками.
- product — лише "Atlas", "Core" або "Desk".
- priority — лише "високий", "середній" або "низький".
- Використовуй подвійні лапки.
- Не додавай Markdown, блоки коду, коментарі, пояснення або інші ключі.
- Не виводь міркувань. Виведи лише кінцевий JSON-об'єкт.


Приклади:

Текст: <<<Desk не відкриває звернення — оператори не можуть працювати.>>>
Відповідь: {"product":"Desk","priority":"високий"}

Текст: <<<Пошук у Atlas став повільнішим, але знаходить правильні записи.>>>
Відповідь: {"product":"Atlas","priority":"середній"}

Текст: <<<Було б зручно додати в Core нову тему оформлення — це nice-to-have.>>>
Відповідь: {"product":"Core","priority":"низький"}
""".strip()


# 4. # 4. Виклик моделі для кожного тексту та вивід результатів.
allowed_products = {"Atlas", "Core", "Desk"}
allowed_priorities = {"високий", "середній", "низький"}

for index, row in enumerate(data, start=1):
    t = row["text"]

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": PROMPT.strip()},
            {
                "role": "user",
                "content": (
                    "Проаналізуй відгук і поверни ТІЛЬКИ JSON "
                    "з ключами product та priority.\n"
                    f"Текст: <<<{t}>>>"
                ),
            },
        ],
        temperature=0,
    )

    result = resp.choices[0].message.content

    # Виводимо оригінальну відповідь моделі.
    print(result)

{"product":"Atlas","priority":"високий"}
{"product":"Core","priority":"середній"}
{"product":"Desk","priority":"низький"}
{"product":"Core","priority":"високий"}
{"product":"Atlas","priority":"високий"}


In [ ]:
# Завдання 3. Генерація заголовка та визначення категорії оголошення

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI


# 1. Завантаження ключа.
env_path = Path.cwd() / ".env"

if not env_path.is_file():
    raise FileNotFoundError(f"Не знайдено файл .env у папці: {Path.cwd()}")

load_dotenv(dotenv_path=env_path)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError(
        "Не знайдено OPENROUTER_API_KEY у змінних середовища або .env."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    timeout=60.0,
    max_retries=0,
)

MODEL = "openai/gpt-oss-120b"


# 2. Дані для аналізу: три стовпчики з таблиці.

data = [
    {
        "text": "Вийшла версія 2.0 з офлайн-режимом і новим дизайном.",
        "category": "реліз",
        "title_example": "Реліз 2.0: офлайн і новий дизайн",
    },
    {
        "text": "У частини користувачів не відкривається профіль після апдейту; працюємо над виправленням.",
        "category": "інцидент",
        "title_example": "Проблеми з профілем після оновлення",
    },
    {
        "text": "Збираємо відгуки про новий модуль аналітики.",
        "category": "рекомендація",
        "title_example": "Поділіться думкою про аналітику",
    },
    {
        "text": "На вихідних сервіс у режимі планового обслуговування.",
        "category": "інцидент",
        "title_example": "Планове обслуговування у вихідні",
    },
    {
        "text": "Рекомендуємо вмикати двофакторну автентифікацію.",
        "category": "рекомендація",
        "title_example": "Увімкніть двофакторну автентифікацію",
    },
]


# 3. Інструкція для моделі.
PROMPT = """
Працюй як редактор коротких українськомовних оголошень.

Завдання:
На основі вхідного тексту створи короткий заголовок українською
та визнач категорію оголошення.

Правила для title:
- Заголовок має містити від 1 до 8 слів включно.
- Словом вважай непорожню послідовність символів,
  відокремлену від інших пробілами.
- Прийменники та сполучники також рахуються як слова.
- Точно передавай головну думку вхідного тексту.
- Не додавай фактів, причин, строків чи обіцянок, яких немає в тексті.
- Зберігай важливі номери версій та назви функцій, якщо вони згадані.
- Не додавай службових вступів на кшталт "Оголошення:",
  "Заголовок:", "Увага:".
- Не додавай емодзі, перенесень рядків або крапки в кінці заголовка.
- Крапка всередині номера версії та дефіс у слові дозволені.
- Не використовуй штучне злиття слів для обходу ліміту.

Правила для category:
- "реліз": повідомлення про вихід або доступність нової версії,
  функції, оновлення чи нового дизайну.
- "інцидент": повідомлення про несправність, збій, недоступність,
  а також планове обслуговування сервісу.
  У межах цього завдання планове обслуговування завжди
  належить до категорії "інцидент".
- "рекомендація": порада, заклик до дії, прохання поділитися
  думкою або повідомлення про збір відгуків.
  Збір відгуків про новий модуль сам по собі не є релізом.

Визначай категорію за основним змістом повідомлення.
Згадка оновлення як причини несправності не робить повідомлення релізом.
Не класифікуй за окремими словами без урахування контексту.

Правила формату:
- Поверни рівно один JSON-об'єкт із рівно двома ключами:
  "title" і "category".
- Обидва значення мають бути рядками.
- category — лише "реліз", "інцидент" або "рекомендація".
- Використовуй подвійні лапки.
- Не додавай Markdown, блоки коду, коментарі, пояснення або інші ключі.
- Не виводь міркувань. Виведи лише кінцевий JSON-об'єкт.
- Перед відповіддю перевір, що заголовок містить не більше 8 слів.
- Не виводь ніяких інших символів, окрім JSON-об'єкта.

""".strip()


# 4. Виклик моделі для кожного тексту та вивід результатів.
for row in data:
    t = row["text"]

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": PROMPT.strip()},
            {
                "role": "user",
                "content": (
                    "Створи заголовок до 8 слів і визнач категорію. "
                    "Поверни ТІЛЬКИ JSON з ключами title та category.\n"
                    f"Текст: <<<{t}>>>"
                ),
            },
        ],
        temperature=0,
    )

    result = resp.choices[0].message.content
    print(result)

{"title":"Вийшла версія 2.0 з офлайн-режимом і новим дизайном","category":"реліз"}
{"title":"Профіль не відкривається після апдейту","category":"інцидент"}
{"title":"Збираємо відгуки про новий модуль аналітики","category":"рекомендація"}
{"title":"Сервіс у режимі планового обслуговування","category":"інцидент"}
{"title":"Вмикайте двофакторну автентифікацію","category":"рекомендація"}
